# NER Fine-tuning Notebook — Kaggle GPU edition — XLM-R / PhoBERT + Macro F1 + HP Search + ONNX INT8

Notebook này là bản chuyển từ Colab sang Kaggle cho pipeline:

**Canonical Dataset → Tokenization Strategy → Experiment Metrics → HP Search → Final Train → ONNX INT8 Export**

Điểm đã đổi cho Kaggle:

1. Không dùng `google.colab.files`.
2. Dataset được đọc tự động từ `/kaggle/input`.
3. Model/artifact được lưu vào `/kaggle/working`.
4. File zip kết quả nằm trong `/kaggle/working/artifacts`.

Trước khi chạy trên Kaggle:

- Vào **Settings → Accelerator** và chọn GPU.
- Bật **Internet** nếu notebook cần tải model/package từ Hugging Face hoặc pip.
- Vào **Add Data** và gắn dataset chứa `processed_for_colab.zip`, hoặc dataset chứa trực tiếp `train.jsonl`, `val.jsonl`, `test.jsonl`.


## Cell 1 — Cài thư viện cho Kaggle "an toàn"

Chức năng:
- Chỉ cài những package còn thiếu, không dùng `pip install -U` để tránh làm xáo trộn môi trường CUDA/Dask có sẵn của Kaggle.
- Cài các package cần cho Hugging Face training và Optuna.
- ONNX/Optimum để export được để ở chế độ tùy chọn, vì bước train model không cần ONNX.

Nếu bạn chỉ cần train model, cứ để `INSTALL_ONNX_EXPORT_LIBS = False`. Khi muốn chạy các cell export ONNX ở cuối notebook, đổi biến này thành `True` rồi chạy lại Cell 1.


In [ ]:
# ============================================================
# CELL 1 — Cài thư viện cho Kaggle, tránh conflict CUDA/Dask
# ============================================================
#
# Vì Kaggle đã cài sẵn torch + CUDA + nhiều package hệ thống,
# KHÔNG dùng `pip install -U` ở đây. Việc upgrade hàng loạt dễ tạo warning kiểu:
#   dask-cuda requires cuda-core==0.3.*, but you have cuda-core 1.0.1
#   dask-cuda requires numba-cuda<0.23.0, but you have numba-cuda 0.30.2
# Những package đó không cần cho notebook NER này, nên ta không đụng vào chúng.

import sys
import subprocess
import importlib.util
import os

os.environ["PIP_ROOT_USER_ACTION"] = "ignore"
os.environ["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"

# Bật True chỉ khi bạn muốn chạy các cell export/quantize ONNX ở cuối notebook.
# Nếu mục tiêu hiện tại là train model trên GPU Kaggle, để False để môi trường ổn định hơn.
INSTALL_ONNX_EXPORT_LIBS = False

TRAINING_PACKAGES = [
    ("transformers", "transformers>=4.44,<4.53"),
    ("datasets", "datasets>=2.20,<3.0"),
    ("evaluate", "evaluate>=0.4.2,<0.5"),
    ("seqeval", "seqeval>=1.2.2,<1.3"),
    ("accelerate", "accelerate>=0.30,<2.0"),
    ("optuna", "optuna>=3.6,<5"),
    ("underthesea", "underthesea>=6.8,<7"),
]

ONNX_PACKAGES = [
    ("optimum", "optimum[onnxruntime]>=1.21,<1.26"),
    ("onnxruntime", "onnxruntime>=1.18,<1.23"),
    ("onnx", "onnx>=1.16,<1.19"),
]


def module_exists(module_name: str) -> bool:
    return importlib.util.find_spec(module_name) is not None


def run_pip_install(packages):
    """Install packages without upgrading Kaggle's CUDA stack."""
    if not packages:
        print("Không có package mới cần cài.")
        return

    cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-cache-dir",
        "--upgrade-strategy",
        "only-if-needed",
        *packages,
    ]

    print("Đang cài:", ", ".join(packages))
    proc = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    output = (proc.stdout or "") + (proc.stderr or "")

    # Nếu pip thật sự fail thì in toàn bộ log để debug.
    if proc.returncode != 0:
        print(output)
        raise RuntimeError("pip install failed")

    # Kaggle đôi khi in warning conflict từ các package hệ thống không liên quan.
    # Lọc các dòng warning đã biết để output sạch hơn; không ẩn lỗi thật vì returncode đã check ở trên.
    ignored_markers = [
        "pip's dependency resolver does not currently take into account",
        "google-adk",
        "dask-cuda",
        "cuda-core",
        "numba-cuda",
    ]
    important_lines = [
        line for line in output.splitlines()
        if line.strip() and not any(marker in line for marker in ignored_markers)
    ]
    if important_lines:
        print("\n".join(important_lines))


def ensure_packages(package_specs):
    missing_specs = []
    for module_name, pip_spec in package_specs:
        if module_exists(module_name):
            print(f"✓ Đã có {module_name}")
        else:
            print(f"• Thiếu {module_name} → sẽ cài {pip_spec}")
            missing_specs.append(pip_spec)
    run_pip_install(missing_specs)


ensure_packages(TRAINING_PACKAGES)

if INSTALL_ONNX_EXPORT_LIBS:
    ensure_packages(ONNX_PACKAGES)
else:
    print("\nBỏ qua ONNX/Optimum ở Cell 1. Train model vẫn chạy bình thường.")
    print("Khi cần chạy các cell export ONNX cuối notebook, đổi INSTALL_ONNX_EXPORT_LIBS = True rồi chạy lại Cell 1.")

# Check version nhanh sau khi cài.
import transformers
import datasets
import evaluate
import optuna

print("\n✅ Cell 1 xong")
print("transformers:", transformers.__version__)
print("datasets    :", datasets.__version__)
print("evaluate    :", evaluate.__version__)
print("optuna      :", optuna.__version__)


## Cell 2 — Kiểm tra runtime GPU/CPU

Chức năng:
- Kiểm tra Kaggle notebook có nhận GPU chưa.
- In version của PyTorch/Transformers để debug khi có lỗi version.


In [ ]:
import os
import torch
import transformers

print("Torch version:", torch.__version__)
print("Transformers version:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Kaggle working dir exists:", os.path.exists("/kaggle/working"))
print("Kaggle input dir exists  :", os.path.exists("/kaggle/input"))

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    !nvidia-smi
else:
    print("Bạn đang chạy CPU. Hãy vào Settings → Accelerator → chọn GPU để fine-tune nhanh hơn.")


## Cell 3 — Cấu hình thí nghiệm

Chức năng:
- Chọn model family: `xlmr` hoặc `phobert`.
- Chọn entity types.
- Cấu hình max length, seed, số trial HP search.
- Mặc định để `RUN_HP_SEARCH = False`; khi baseline ổn thì đổi thành `True`.

In [ ]:
from pathlib import Path

# ============================================================
# 0) Kaggle paths
# ============================================================
# Kaggle:
# - /kaggle/input   : nơi chứa dataset đã Add Data, read-only
# - /kaggle/working : nơi ghi model, log, artifact, file zip
#
# Nếu chạy local/Jupyter thường, WORKING_ROOT sẽ fallback về thư mục hiện tại.
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()

print("KAGGLE_INPUT_ROOT:", KAGGLE_INPUT_ROOT)
print("WORKING_ROOT     :", WORKING_ROOT)

# ============================================================
# 1) Chọn model family
# ============================================================
# "xlmr"    : khuyến nghị chạy trước, dùng raw text + offset_mapping.
# "phobert" : chạy sau để so sánh, cần word segmentation tiếng Việt.
MODEL_FAMILY = "xlmr"  # đổi thành "phobert" nếu muốn thử PhoBERT

MODEL_REGISTRY = {
    "xlmr": {
        "model_name": "xlm-roberta-base",
        "output_dir": str(WORKING_ROOT / "runs" / "xlmr_ner_final"),
        "hp_dir": str(WORKING_ROOT / "runs" / "xlmr_hp_search"),
        "onnx_fp32_dir": str(WORKING_ROOT / "onnx" / "xlmr_fp32"),
        "onnx_int8_dir": str(WORKING_ROOT / "onnx" / "xlmr_int8"),
    },
    "phobert": {
        # base-v2 thường là lựa chọn tốt hơn base cũ.
        "model_name": "vinai/phobert-base-v2",
        "output_dir": str(WORKING_ROOT / "runs" / "phobert_ner_final"),
        "hp_dir": str(WORKING_ROOT / "runs" / "phobert_hp_search"),
        "onnx_fp32_dir": str(WORKING_ROOT / "onnx" / "phobert_fp32"),
        "onnx_int8_dir": str(WORKING_ROOT / "onnx" / "phobert_int8"),
    },
}

CFG = MODEL_REGISTRY[MODEL_FAMILY]
MODEL_NAME = CFG["model_name"]

# ============================================================
# 2) Entity schema
# ============================================================
ENTITY_TYPES = ["PER", "ADDR", "NOTE"]

# BIO labels chuẩn IOB2.
DEFAULT_LABEL_LIST = [
    "O",
    "B-PER", "I-PER",
    "B-ADDR", "I-ADDR",
    "B-NOTE", "I-NOTE",
]

# ============================================================
# 3) Training config
# ============================================================
MAX_LENGTH = 256
SEED = 42

# Smoke test giúp bắt lỗi pipeline trước khi train full.
RUN_SMOKE_TEST = True

# HP search khá tốn GPU. Chạy sau khi smoke test và baseline đã ổn.
RUN_HP_SEARCH = False
HP_N_TRIALS = 20

# Test set chỉ nên evaluate sau cùng, tránh leakage.
RUN_FINAL_TEST = True

print("MODEL_FAMILY:", MODEL_FAMILY)
print("MODEL_NAME:", MODEL_NAME)
print("CFG:")
for k, v in CFG.items():
    print(f" - {k}: {v}")


## Cell 4 — Import và set seed

Chức năng:
- Import các thư viện dùng xuyên suốt notebook.
- Set seed để kết quả ổn định hơn giữa các lần chạy.

In [ ]:
import os
import json
import random
import zipfile
import re
import time
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed,
)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Path(CFG["output_dir"]).parent.mkdir(parents=True, exist_ok=True)
Path(CFG["hp_dir"]).parent.mkdir(parents=True, exist_ok=True)
Path(CFG["onnx_fp32_dir"]).parent.mkdir(parents=True, exist_ok=True)
Path(CFG["onnx_int8_dir"]).parent.mkdir(parents=True, exist_ok=True)


## Cell 5 — Đọc dataset từ Kaggle Input

Chức năng:
- Tự tìm `processed_for_colab.zip` trong `/kaggle/input`.
- Nếu không có zip, tự tìm trực tiếp `train.jsonl`, `val.jsonl`, `test.jsonl`.
- Chuẩn hóa về thư mục writable `/kaggle/working/ner_dataset_normalized`.

Cách dùng trên Kaggle:
- Vào **Add Data** và gắn dataset chứa `processed_for_colab.zip`, hoặc gắn dataset chứa trực tiếp 3 file JSONL.


In [ ]:
# ============================================================
# CELL 5 — Đọc dataset từ Kaggle Input và chuẩn hóa train/val/test
# ============================================================
#
# Input hỗ trợ:
# 1) Dataset Kaggle chứa processed_for_colab.zip
# 2) Dataset Kaggle chứa trực tiếp train.jsonl / val.jsonl / test.jsonl
# 3) Zip có path kiểu Windows, ví dụ processed\train.jsonl
#
# Output chuẩn hóa:
#   /kaggle/working/ner_dataset_normalized/train.jsonl
#   /kaggle/working/ner_dataset_normalized/val.jsonl
#   /kaggle/working/ner_dataset_normalized/test.jsonl
#
# Các cell sau chỉ cần dùng biến DATA_DIR.

from pathlib import Path
import zipfile
import shutil

ZIP_NAME = "processed_for_colab.zip"

# Nếu muốn chỉ định thủ công, sửa dòng dưới, ví dụ:
# DATASET_ZIP_PATH = Path("/kaggle/input/my-dataset/processed_for_colab.zip")
DATASET_ZIP_PATH = None

DATA_DIR = WORKING_ROOT / "ner_dataset_normalized"

# ------------------------------------------------------------
# Utility
# ------------------------------------------------------------

def normalize_zip_name(name):
    return name.replace("\\", "/").lower()

def split_candidates(split):
    if split == "train":
        return ["train.jsonl"]
    if split == "val":
        return ["val.jsonl", "validation.jsonl", "dev.jsonl"]
    if split == "test":
        return ["test.jsonl"]
    raise ValueError("split phải là train, val hoặc test")

def pick_entry_by_split(entries, split):
    """
    Chọn file jsonl tương ứng với split.

    Hỗ trợ:
    - train.jsonl
    - val.jsonl
    - validation.jsonl
    - dev.jsonl
    - test.jsonl
    - path kiểu processed/train.jsonl
    - path kiểu processed\\train.jsonl
    """
    candidates = split_candidates(split)
    matches = []

    for entry in entries:
        normalized = normalize_zip_name(str(entry))
        basename = normalized.split("/")[-1]

        for priority, expected_name in enumerate(candidates):
            # Ưu tiên match chính xác basename.
            if basename == expected_name:
                matches.append((0, priority, len(normalized), entry))
                break

            # Match mềm nếu tên file có chứa train/val/test.
            stem = expected_name.replace(".jsonl", "")
            if stem in basename:
                matches.append((1, priority, len(normalized), entry))
                break

    if not matches:
        return None

    matches = sorted(matches, key=lambda x: (x[0], x[1], x[2]))
    return matches[0][3]

def count_lines(path):
    with open(path, "r", encoding="utf-8") as f:
        return sum(1 for _ in f)

def reset_data_dir():
    if DATA_DIR.exists():
        shutil.rmtree(DATA_DIR)
    DATA_DIR.mkdir(parents=True, exist_ok=True)

def copy_file_to_data_dir(src_path, split):
    dst = DATA_DIR / f"{split}.jsonl"
    shutil.copyfile(src_path, dst)
    return dst

def copy_zip_entry_to_file(zip_path, entry_name, output_path):
    """
    Copy một file bên trong zip ra output_path.
    Không dùng extractall để tránh lỗi path Windows/Linux.
    """
    with zipfile.ZipFile(zip_path, "r") as zf:
        with zf.open(entry_name, "r") as src:
            data = src.read()
    output_path.write_bytes(data)

def zip_contains_required_splits(zip_path):
    try:
        with zipfile.ZipFile(zip_path, "r") as zf:
            names = zf.namelist()
    except Exception:
        return False

    jsonl_entries = [name for name in names if normalize_zip_name(name).endswith(".jsonl")]
    return all(pick_entry_by_split(jsonl_entries, split) is not None for split in ["train", "val", "test"])

# ------------------------------------------------------------
# 1. Tìm dataset
# ------------------------------------------------------------

print("KAGGLE_INPUT_ROOT:", KAGGLE_INPUT_ROOT)
print("WORKING_ROOT     :", WORKING_ROOT)

if DATASET_ZIP_PATH is not None:
    DATASET_ZIP_PATH = Path(DATASET_ZIP_PATH)
    if not DATASET_ZIP_PATH.exists():
        raise FileNotFoundError(f"DATASET_ZIP_PATH không tồn tại: {DATASET_ZIP_PATH}")
    zip_candidates = [DATASET_ZIP_PATH]
else:
    if KAGGLE_INPUT_ROOT.exists():
        # Ưu tiên đúng tên processed_for_colab.zip.
        named_zips = sorted(KAGGLE_INPUT_ROOT.rglob(ZIP_NAME))
        other_zips = sorted(p for p in KAGGLE_INPUT_ROOT.rglob("*.zip") if p.name != ZIP_NAME)
        zip_candidates = named_zips + other_zips
    else:
        zip_candidates = []

print("\nZip candidates:")
for p in zip_candidates[:20]:
    print(" -", p)

ZIP_PATH = None
for p in zip_candidates:
    if zip_contains_required_splits(p):
        ZIP_PATH = p
        break

# ------------------------------------------------------------
# 2A. Nếu tìm thấy zip, đọc từ zip
# ------------------------------------------------------------

if ZIP_PATH is not None:
    print("\nĐang dùng ZIP_PATH:", ZIP_PATH)

    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zip_names = zf.namelist()

    print("\nMột số file trong zip:")
    for name in zip_names[:50]:
        print(" -", name)

    jsonl_entries = [
        name for name in zip_names
        if normalize_zip_name(name).endswith(".jsonl")
    ]

    print("\nTìm thấy các file .jsonl trong zip:")
    for name in jsonl_entries:
        print(" -", name)

    TRAIN_ENTRY = pick_entry_by_split(jsonl_entries, "train")
    VAL_ENTRY = pick_entry_by_split(jsonl_entries, "val")
    TEST_ENTRY = pick_entry_by_split(jsonl_entries, "test")

    print("\nKết quả nhận diện split trong zip:")
    print("TRAIN_ENTRY:", TRAIN_ENTRY)
    print("VAL_ENTRY  :", VAL_ENTRY)
    print("TEST_ENTRY :", TEST_ENTRY)

    missing = []
    if TRAIN_ENTRY is None:
        missing.append("train.jsonl")
    if VAL_ENTRY is None:
        missing.append("val.jsonl / validation.jsonl / dev.jsonl")
    if TEST_ENTRY is None:
        missing.append("test.jsonl")

    if missing:
        raise FileNotFoundError(
            "Không tìm thấy các split sau trong zip: "
            + ", ".join(missing)
        )

    reset_data_dir()

    copy_zip_entry_to_file(ZIP_PATH, TRAIN_ENTRY, DATA_DIR / "train.jsonl")
    copy_zip_entry_to_file(ZIP_PATH, VAL_ENTRY, DATA_DIR / "val.jsonl")
    copy_zip_entry_to_file(ZIP_PATH, TEST_ENTRY, DATA_DIR / "test.jsonl")

# ------------------------------------------------------------
# 2B. Nếu không tìm thấy zip, thử tìm trực tiếp JSONL trong /kaggle/input
# ------------------------------------------------------------

else:
    print("\nKhông tìm thấy zip hợp lệ. Thử tìm trực tiếp train/val/test JSONL trong /kaggle/input...")

    if not KAGGLE_INPUT_ROOT.exists():
        raise FileNotFoundError(
            "Không thấy /kaggle/input. Nếu đang chạy local, hãy đặt DATASET_ZIP_PATH thủ công."
        )

    jsonl_files = sorted(KAGGLE_INPUT_ROOT.rglob("*.jsonl"))

    print("\nJSONL candidates:")
    for p in jsonl_files[:50]:
        print(" -", p)

    TRAIN_FILE = pick_entry_by_split(jsonl_files, "train")
    VAL_FILE = pick_entry_by_split(jsonl_files, "val")
    TEST_FILE = pick_entry_by_split(jsonl_files, "test")

    print("\nKết quả nhận diện split trực tiếp:")
    print("TRAIN_FILE:", TRAIN_FILE)
    print("VAL_FILE  :", VAL_FILE)
    print("TEST_FILE :", TEST_FILE)

    missing = []
    if TRAIN_FILE is None:
        missing.append("train.jsonl")
    if VAL_FILE is None:
        missing.append("val.jsonl / validation.jsonl / dev.jsonl")
    if TEST_FILE is None:
        missing.append("test.jsonl")

    if missing:
        raise FileNotFoundError(
            "Không tìm thấy dataset trong /kaggle/input.\n"
            "Hãy vào Add Data và gắn dataset chứa processed_for_colab.zip "
            "hoặc 3 file train.jsonl / val.jsonl / test.jsonl.\n"
            "Thiếu: " + ", ".join(missing)
        )

    reset_data_dir()

    copy_file_to_data_dir(TRAIN_FILE, "train")
    copy_file_to_data_dir(VAL_FILE, "val")
    copy_file_to_data_dir(TEST_FILE, "test")

# ------------------------------------------------------------
# 3. Kiểm tra output chuẩn hóa
# ------------------------------------------------------------

print("\nĐã chuẩn hóa dataset vào:", DATA_DIR)
print(" -", DATA_DIR / "train.jsonl")
print(" -", DATA_DIR / "val.jsonl")
print(" -", DATA_DIR / "test.jsonl")

print("\nSố dòng:")
print("train:", count_lines(DATA_DIR / "train.jsonl"))
print("val  :", count_lines(DATA_DIR / "val.jsonl"))
print("test :", count_lines(DATA_DIR / "test.jsonl"))

print("\n✅ Cell 5 hoàn tất. Các cell sau dùng biến DATA_DIR.")


## Cell 6 — Đọc JSONL và label map

Chức năng:
- Đọc `train.jsonl`, `val.jsonl`, `test.jsonl`.
- Đọc `label2id.json` nếu có.
- Nếu không có `label2id.json`, tự tạo label map chuẩn IOB2.

In [ ]:
def read_jsonl(path: Path):
    """Đọc file JSONL, bỏ qua dòng rỗng."""
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

train_rows = read_jsonl(DATA_DIR / "train.jsonl")
val_rows = read_jsonl(DATA_DIR / "val.jsonl")
test_rows = read_jsonl(DATA_DIR / "test.jsonl")

label2id_path = DATA_DIR / "label2id.json"
if label2id_path.exists():
    raw_label2id = json.load(open(label2id_path, "r", encoding="utf-8"))
    # Đảm bảo value là int.
    label2id = {str(k): int(v) for k, v in raw_label2id.items()}
else:
    label2id = {label: i for i, label in enumerate(DEFAULT_LABEL_LIST)}

id2label = {int(v): str(k) for k, v in label2id.items()}

print("Số mẫu:")
print("train:", len(train_rows))
print("validation:", len(val_rows))
print("test:", len(test_rows))
print()
print("label2id:", label2id)

## Cell 7 — Validate dữ liệu gốc

Chức năng:
- Kiểm tra span entity có nằm trong text không.
- Kiểm tra entity có bị overlap không.
- Kiểm tra label có thuộc `PER`, `ADDR`, `NOTE` không.
- Lọc bỏ mẫu lỗi để tránh crash khi tokenize/train.

Ghi chú:
- Dataset gốc vẫn nên giữ raw text + character span.
- Không sửa dataset gốc để phục vụ riêng một tokenizer nào.

In [ ]:
VALID_LABELS = set(ENTITY_TYPES)

def normalize_entity(ent):
    """Chuẩn hóa entity dict về dạng {start, end, label}."""
    return {
        "start": int(ent["start"]),
        "end": int(ent["end"]),
        "label": str(ent["label"]),
    }

def is_valid_example(ex, verbose=False):
    """Validate một example NER theo character span."""
    text = ex.get("text", "")
    entities = [normalize_entity(e) for e in ex.get("entities", [])]

    prev_end = -1
    for ent in sorted(entities, key=lambda e: e["start"]):
        s, e, label = ent["start"], ent["end"], ent["label"]

        if not (0 <= s < e <= len(text)):
            if verbose:
                print("Invalid span:", ent, "text length:", len(text))
            return False

        if s < prev_end:
            if verbose:
                print("Overlapping span:", ent)
            return False

        if label not in VALID_LABELS:
            if verbose:
                print("Invalid label:", label)
            return False

        prev_end = e

    return True

def clean_rows(rows, split_name):
    """Lọc mẫu lỗi và chuẩn hóa entities."""
    clean = []
    bad = []

    for ex in rows:
        if is_valid_example(ex):
            ex = dict(ex)
            ex["entities"] = [normalize_entity(e) for e in ex.get("entities", [])]
            clean.append(ex)
        else:
            bad.append(ex)

    print(f"{split_name}: kept={len(clean)}, dropped={len(bad)}")
    if bad:
        print("Ví dụ mẫu lỗi đầu tiên:")
        print(json.dumps(bad[0], ensure_ascii=False, indent=2)[:1000])

    return clean

train_rows = clean_rows(train_rows, "train")
val_rows = clean_rows(val_rows, "validation")
test_rows = clean_rows(test_rows, "test")

## Cell 8 — Thống kê nhanh dataset

Chức năng:
- Đếm số entity theo class.
- Xem phân phối độ dài text.
- Giúp phát hiện class hiếm, đặc biệt `NOTE`.

In [ ]:
def dataset_stats(rows, name):
    counter = Counter()
    lengths = []

    for ex in rows:
        lengths.append(len(ex["text"]))
        for ent in ex.get("entities", []):
            counter[ent["label"]] += 1

    return {
        "split": name,
        "n_examples": len(rows),
        "avg_chars": float(np.mean(lengths)) if lengths else 0,
        "p95_chars": float(np.percentile(lengths, 95)) if lengths else 0,
        **{f"n_{label}": counter.get(label, 0) for label in ENTITY_TYPES},
    }

stats_df = pd.DataFrame([
    dataset_stats(train_rows, "train"),
    dataset_stats(val_rows, "validation"),
    dataset_stats(test_rows, "test"),
])

display(stats_df)

## Cell 9 — Load tokenizer theo strategy

Chức năng:
- XLM-R dùng fast tokenizer để lấy `offset_mapping`.
- PhoBERT dùng tokenizer riêng và text phải được word-segmented trước.
- Notebook vẫn giữ chung interface `tokenize_and_align(ex)`.

In [ ]:
if MODEL_FAMILY == "xlmr":
    # XLM-R cần fast tokenizer để map token ↔ character span.
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    if not tokenizer.is_fast:
        raise RuntimeError("XLM-R cần fast tokenizer để dùng offset_mapping.")
elif MODEL_FAMILY == "phobert":
    # PhoBERT nhận input đã word-segmented.
    # Dùng manual tokenization theo word để không phụ thuộc bắt buộc vào word_ids().
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
else:
    raise ValueError(f"Unknown MODEL_FAMILY: {MODEL_FAMILY}")

print("Tokenizer class:", tokenizer.__class__.__name__)
print("Tokenizer is_fast:", getattr(tokenizer, "is_fast", False))
print("model_max_length:", tokenizer.model_max_length)

## Cell 10 — BIO utility functions

Chức năng:
- Convert entity label `PER/ADDR/NOTE` sang BIO label.
- Gán label cho token/word dựa trên character span.
- Dùng chung cho cả XLM-R và PhoBERT.

In [ ]:
ENTITY_TO_BIO = {
    "PER": ("B-PER", "I-PER"),
    "ADDR": ("B-ADDR", "I-ADDR"),
    "NOTE": ("B-NOTE", "I-NOTE"),
}

def bio_label_for_span(unit_start, unit_end, entities):
    """
    Gán BIO label cho một unit, có thể là token hoặc segmented word.

    Logic:
    - Nếu unit nằm hoàn toàn trong entity span → B/I theo vị trí start.
    - Nếu không nằm trong entity nào → O.
    - Cách này bảo thủ, tránh gán nhầm khi unit cắt ngang boundary entity.
    """
    if unit_start is None or unit_end is None or unit_start >= unit_end:
        return "O"

    for ent in entities:
        s, e, label = ent["start"], ent["end"], ent["label"]
        if unit_start >= s and unit_end <= e:
            b_label, i_label = ENTITY_TO_BIO[label]
            return b_label if unit_start == s else i_label

    return "O"

def ensure_iob2(labels):
    """
    Sửa nhẹ chuỗi BIO để inference bớt lỗi:
    - Nếu gặp I-X sau O hoặc sau entity khác loại, đổi thành B-X.
    - Dùng cho output inference, không dùng để 'làm đẹp' metric training.
    """
    fixed = []
    prev_type = None

    for lab in labels:
        if lab == "O" or lab is None:
            fixed.append("O")
            prev_type = None
            continue

        prefix, ent_type = lab.split("-", 1)
        if prefix == "I" and prev_type != ent_type:
            lab = "B-" + ent_type

        fixed.append(lab)
        prev_type = ent_type

    return fixed

## Cell 11 — Tokenization strategy cho XLM-R

Chức năng:
- Tokenize raw text bằng `return_offsets_mapping=True`.
- Dùng offset của từng token để gán BIO label.
- Special tokens được gán `-100` để loss bỏ qua.

In [ ]:
def tokenize_and_align_xlmr(ex):
    """Tokenize + align labels cho XLM-R bằng character offsets."""
    enc = tokenizer(
        ex["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        return_offsets_mapping=True,
    )

    labels = []
    for token_start, token_end in enc["offset_mapping"]:
        # Special tokens thường có offset (0, 0), cần ignore trong loss.
        if token_start == token_end:
            labels.append(-100)
            continue

        bio_label = bio_label_for_span(token_start, token_end, ex.get("entities", []))
        labels.append(label2id[bio_label])

    # Không đưa offset_mapping vào model.
    enc.pop("offset_mapping")
    enc["labels"] = labels
    return enc

## Cell 12 — Tokenization strategy cho PhoBERT

Chức năng:
- Word-segment raw Vietnamese text bằng `underthesea`.
- Map segmented word trở lại character span gốc.
- Tokenize từng segmented word thủ công để gán label cho first subword.
- Cách này chạy được cả khi PhoBERT tokenizer không có fast `word_ids()`.

Lưu ý:
- PhoBERT yêu cầu input đã word-segmented, ví dụ `Hà Nội` → `Hà_Nội`.
- Alignment PhoBERT phức tạp hơn XLM-R, nên chỉ chạy sau khi XLM-R baseline đã ổn.

In [ ]:
# ============================================================
# FIX — candidate surface robust hơn cho dấu "_" của PhoBERT
# ============================================================
#
# Vấn đề:
# - PhoBERT/underthesea dùng "_" để nối word-segment.
# - Nhưng "_" có thể tương ứng với:
#     1. khoảng trắng trong raw text
#     2. không có ký tự nào trong raw text
#
# Ví dụ:
# - "Trần_Hưng_Đạo"   -> "Trần Hưng Đạo"
# - "9A-11_A_Dương"   -> "9A-11A Dương"
# - "PHONE_]"         -> "PHONE]"
# - "Q.Gò_Vấp"        -> "Q.Gò Vấp"

import re
from itertools import product


def _candidate_surfaces_from_segmented_word(w, max_enum_underscores=8):
    """
    Sinh các dạng surface có thể xuất hiện trong raw text.

    Với mỗi dấu "_", thử cả 2 khả năng:
    - "_" -> " "
    - "_" -> ""

    Điều này xử lý được case:
    - 9A-11_A_Dương -> 9A-11A Dương
    """

    candidates = []

    def add(x):
        if x and x not in candidates:
            candidates.append(x)

    # 1. Giữ nguyên token segmented
    add(w)

    # 2. Các biến thể phổ biến
    add(w.replace("_", " "))
    add(w.replace("_", ""))

    # 3. Sửa artefact "_" cạnh dấu câu
    punct_fixed = re.sub(r"_([^\wÀ-ỹ])", r"\1", w)
    punct_fixed = re.sub(r"([^\wÀ-ỹ])_", r"\1", punct_fixed)

    add(punct_fixed)
    add(punct_fixed.replace("_", " "))
    add(punct_fixed.replace("_", ""))

    # 4. Sinh tổ hợp cho từng dấu "_":
    #    mỗi "_" có thể là " " hoặc "".
    #
    # Ví dụ:
    #   9A-11_A_Dương
    #     "_" thứ 1 -> ""
    #     "_" thứ 2 -> " "
    #   => 9A-11A Dương
    underscore_count = w.count("_")

    if 0 < underscore_count <= max_enum_underscores:
        parts = w.split("_")

        for choices in product([" ", ""], repeat=underscore_count):
            out = parts[0]

            for sep, part in zip(choices, parts[1:]):
                out += sep + part

            add(out)

    # 5. Làm lại tổ hợp trên bản đã sửa punctuation
    underscore_count_pf = punct_fixed.count("_")

    if punct_fixed != w and 0 < underscore_count_pf <= max_enum_underscores:
        parts = punct_fixed.split("_")

        for choices in product([" ", ""], repeat=underscore_count_pf):
            out = parts[0]

            for sep, part in zip(choices, parts[1:]):
                out += sep + part

            add(out)

    # 6. Ưu tiên candidate dài hơn trước nếu cùng vị trí match.
    #    Nhưng vẫn giữ thứ tự tương đối ban đầu.
    return candidates

In [ ]:
# ============================================================
# FIX — xử lý ký tự Unicode dễ nhầm: Ð/ð ↔ Đ/đ
# ============================================================

import unicodedata


def normalize_vietnamese_confusables(s):
    """
    Chuẩn hóa các ký tự nhìn giống tiếng Việt nhưng khác Unicode.

    Ví dụ:
    - Ð -> Đ
    - ð -> đ

    Lưu ý:
    - Chỉ dùng để so sánh/matching.
    - Không thay đổi raw text gốc.
    """

    return (
        s.replace("Ð", "Đ")
         .replace("ð", "đ")
    )


def strip_vietnamese_accents(s):
    """
    Bỏ dấu tiếng Việt để so sánh mềm.

    Ví dụ:
    - "thuỷ" -> "thuy"
    - "thủy" -> "thuy"
    - "Ðặng" -> "dang"
    - "Đặng" -> "dang"
    """

    s = normalize_vietnamese_confusables(s)

    s = unicodedata.normalize("NFD", s)
    s = "".join(ch for ch in s if unicodedata.category(ch) != "Mn")

    s = s.replace("đ", "d").replace("Đ", "D")

    return s


def find_surface_robust(chunk_text, surface, cursor):
    """
    Tìm surface trong chunk_text từ vị trí cursor.

    Thứ tự:
    1. Exact match.
    2. NFC match.
    3. Confusable-normalized match, ví dụ Ðặng ↔ Đặng.
    4. Accent-insensitive match, ví dụ thuỷ ↔ thủy.
    """

    # 1. Exact match
    idx = chunk_text.find(surface, cursor)
    if idx != -1:
        return idx

    # 2. Unicode NFC match
    normalized_chunk = unicodedata.normalize("NFC", chunk_text)
    normalized_surface = unicodedata.normalize("NFC", surface)

    idx = normalized_chunk.find(normalized_surface, cursor)
    if idx != -1:
        return idx

    # 3. Confusable-normalized match
    conf_chunk = normalize_vietnamese_confusables(chunk_text)
    conf_surface = normalize_vietnamese_confusables(surface)

    idx = conf_chunk.find(conf_surface, cursor)
    if idx != -1:
        return idx

    # 4. Accent-insensitive fallback
    target = strip_vietnamese_accents(surface).lower()
    window_len = len(surface)

    for i in range(cursor, len(chunk_text) - window_len + 1):
        window = chunk_text[i:i + window_len]

        if strip_vietnamese_accents(window).lower() == target:
            return i

    return None

## Cell 13 — Chọn `tokenize_and_align` theo model family

Chức năng:
- Áp dụng Strategy pattern.
- Các cell sau không cần biết đang chạy XLM-R hay PhoBERT.

In [ ]:
def tokenize_and_align(ex):
    if MODEL_FAMILY == "xlmr":
        return tokenize_and_align_xlmr(ex)
    elif MODEL_FAMILY == "phobert":
        return tokenize_and_align_phobert(ex)
    else:
        raise ValueError(f"Unknown MODEL_FAMILY: {MODEL_FAMILY}")

## Cell 14 — Preview alignment trên 1 mẫu

Chức năng:
- In token/subword và label tương ứng.
- Cell này rất quan trọng để kiểm tra alignment trước khi train.

In [ ]:
sample_idx = 0
sample = tokenize_and_align(train_rows[sample_idx])

tokens = tokenizer.convert_ids_to_tokens(sample["input_ids"])
labels = [
    id2label[int(x)] if int(x) != -100 else "IGN"
    for x in sample["labels"]
]

print("Raw text:")
print(train_rows[sample_idx]["text"])
print()
print("Token alignment preview:")
for tok, lab in list(zip(tokens, labels))[:120]:
    print(f"{tok:20s} {lab}")

## Cell 15 — Tạo DatasetDict và tokenize từng split

Chức năng:
- Tạo `DatasetDict(train/validation/test)`.
- Tokenize từng split riêng.
- `remove_columns=dataset.column_names` dùng column của chính split đó để tránh lỗi lệch schema như cột `variation`.

In [ ]:
raw_ds = DatasetDict({
    "train": Dataset.from_list(train_rows),
    "validation": Dataset.from_list(val_rows),
    "test": Dataset.from_list(test_rows),
})

tokenized_ds = DatasetDict({
    split: dataset.map(
        tokenize_and_align,
        remove_columns=dataset.column_names,
        desc=f"Tokenizing {split}",
    )
    for split, dataset in raw_ds.items()
})

print(tokenized_ds)
print("Features train:", tokenized_ds["train"].features)

## Cell 16 — Data collator

Chức năng:
- Pad động theo batch.
- Pad `labels` bằng `-100`, đúng chuẩn token classification.

In [ ]:
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    label_pad_token_id=-100,
)

## Cell 17 — Compute metrics: strict entity-level micro/macro F1

Chức năng:
- Dùng `seqeval` để đánh giá NER ở entity/chunk level.
- Trả về:
  - `f1`: micro entity F1.
  - `f1_macro`: macro F1 trung bình không trọng số qua `PER`, `ADDR`, `NOTE`.
  - `f1_PER`, `f1_ADDR`, `f1_NOTE`.
- Dùng `f1_macro` làm metric chính để chọn best model.

In [ ]:
import evaluate

seqeval = evaluate.load("seqeval")

def decode_predictions_and_labels(predictions, labels):
    """Chuyển logits/label ids thành list BIO labels, bỏ vị trí -100."""
    pred_ids = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for pred_seq, label_seq in zip(pred_ids, labels):
        cur_preds = []
        cur_labels = []

        for pred_id, label_id in zip(pred_seq, label_seq):
            if int(label_id) == -100:
                continue

            cur_preds.append(id2label[int(pred_id)])
            cur_labels.append(id2label[int(label_id)])

        true_predictions.append(cur_preds)
        true_labels.append(cur_labels)

    return true_predictions, true_labels

def compute_metrics(p):
    """Metric chính cho Trainer."""
    predictions, labels = p
    true_predictions, true_labels = decode_predictions_and_labels(predictions, labels)

    # Strict IOB2 giúp đánh giá đúng entity span hơn.
    result = seqeval.compute(
        predictions=true_predictions,
        references=true_labels,
        scheme="IOB2",
        mode="strict",
        zero_division=0,
    )

    # Macro F1: class nào không xuất hiện trong report thì tính F1 = 0.
    per_class_f1 = {
        ent_type: float(result.get(ent_type, {}).get("f1", 0.0))
        for ent_type in ENTITY_TYPES
    }
    macro_f1 = float(np.mean(list(per_class_f1.values())))

    metrics = {
        "precision": float(result.get("overall_precision", 0.0)),
        "recall": float(result.get("overall_recall", 0.0)),
        "f1": float(result.get("overall_f1", 0.0)),       # micro entity F1
        "f1_macro": macro_f1,                             # metric chọn model
        "accuracy": float(result.get("overall_accuracy", 0.0)),
    }

    for ent_type in ENTITY_TYPES:
        metrics[f"f1_{ent_type}"] = per_class_f1[ent_type]
        metrics[f"precision_{ent_type}"] = float(result.get(ent_type, {}).get("precision", 0.0))
        metrics[f"recall_{ent_type}"] = float(result.get(ent_type, {}).get("recall", 0.0))
        metrics[f"support_{ent_type}"] = float(result.get(ent_type, {}).get("number", 0.0))

    return metrics

## Cell 18 — Model factory

Chức năng:
- Tạo model mới từ checkpoint.
- Dùng chung cho smoke test, baseline/final train và Optuna HP search.
- `model_init` rất quan trọng cho HP search vì mỗi trial cần model mới.

In [ ]:
def model_init(trial=None):
    """Factory tạo model token classification mới."""
    return AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(label2id),
        id2label=id2label,
        label2id=label2id,
    )

## Cell 19 — Tiny smoke training

Chức năng:
- Train thử trên 64 mẫu để bắt lỗi pipeline.
- Không dùng kết quả này làm kết quả model.
- Nếu cell này fail, không nên chạy HP search/final train.

In [ ]:
if RUN_SMOKE_TEST:
    tiny_train_size = min(64, len(tokenized_ds["train"]))
    tiny_val_size = min(64, len(tokenized_ds["validation"]))

    tiny_train = tokenized_ds["train"].select(range(tiny_train_size))
    tiny_val = tokenized_ds["validation"].select(range(tiny_val_size))

    smoke_args = TrainingArguments(
        output_dir=f"runs/{MODEL_FAMILY}_smoke_test",
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=1,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=10,
        report_to="none",
        fp16=torch.cuda.is_available(),
        seed=SEED,
    )

    smoke_trainer = Trainer(
        model_init=model_init,
        args=smoke_args,
        train_dataset=tiny_train,
        eval_dataset=tiny_val,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    smoke_trainer.train()
    smoke_metrics = smoke_trainer.evaluate()
    print("Smoke metrics:", smoke_metrics)
else:
    print("Bỏ qua smoke test.")

## Cell 20 — Hyperparameter search bằng Optuna

Chức năng:
- Tìm learning rate, epoch, batch size, weight decay, warmup ratio.
- Objective là `eval_f1_macro`.
- Lưu best hyperparameters ra JSON.

Lưu ý:
- Các key trong `hp_space` phải đúng tên field của `TrainingArguments`, ví dụ `learning_rate`, không đặt alias như `lr`.
- Nếu GPU yếu hoặc dataset lớn, giảm `HP_N_TRIALS`.

In [ ]:
# ============================================================
# CELL A — Cấu hình chạy Optuna HP Search
# ============================================================

RUN_HP_SEARCH = True

# Test nhanh thì để 3 hoặc 5.
# Chạy nghiêm túc thì để 20–30.
HP_N_TRIALS = 10

print("RUN_HP_SEARCH:", RUN_HP_SEARCH)
print("HP_N_TRIALS:", HP_N_TRIALS)

In [ ]:
# ============================================================
# CELL B — Hyperparameter Search bằng Optuna
# ============================================================
#
# Chức năng:
# - Thử nhiều bộ hyperparameter khác nhau.
# - Mỗi trial train model từ đầu.
# - Chọn bộ tốt nhất theo eval_f1_macro.
# - Lưu kết quả vào best_hyperparameters.json.
#
# Output mong muốn:
# - Best run
# - Best hyperparameters
# - File best_hyperparameters.json

from pathlib import Path
import json
import torch

from transformers import TrainingArguments, Trainer

BEST_HP_PATH = Path(CFG["hp_dir"]) / "best_hyperparameters.json"

print("RUN_HP_SEARCH:", RUN_HP_SEARCH)
print("HP_N_TRIALS:", HP_N_TRIALS)
print("BEST_HP_PATH:", BEST_HP_PATH)

if not RUN_HP_SEARCH:
    print("\nRUN_HP_SEARCH=False nên bỏ qua HP search.")
    print("Muốn tìm best HP thì chạy:")
    print("RUN_HP_SEARCH = True")
else:
    hp_args = TrainingArguments(
        output_dir=CFG["hp_dir"],

        # Evaluate sau mỗi epoch để lấy eval_f1_macro.
        eval_strategy="epoch",

        # HP search không cần save checkpoint từng trial.
        save_strategy="no",

        # Không load best model trong HP search.
        load_best_model_at_end=False,

        # Tắt wandb/tensorboard.
        report_to="none",

        logging_steps=50,

        # GPU Kaggle/Colab dùng fp16 được.
        # Nếu bị lỗi fp16, đổi thành False.
        fp16=torch.cuda.is_available(),

        seed=SEED,

        # Default tạm; Optuna sẽ override các key trong hp_space.
        learning_rate=2e-5,
        num_train_epochs=3,
        per_device_train_batch_size=8,
        weight_decay=0.01,
        warmup_ratio=0.1,
    )

    trainer_hp = Trainer(
        # Quan trọng: dùng model_init, không dùng model=...
        # Mỗi trial cần model mới từ cùng pretrained checkpoint.
        model_init=model_init,

        args=hp_args,
        train_dataset=tokenized_ds["train"],
        eval_dataset=tokenized_ds["validation"],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    def hp_space(trial):
        """
        Không nhập từng bộ thủ công ở đây.
        Optuna sẽ tự sample trong các khoảng dưới.
        """

        return {
            "learning_rate": trial.suggest_float(
                "learning_rate",
                1e-5,
                5e-5,
                log=True,
            ),
            "num_train_epochs": trial.suggest_int(
                "num_train_epochs",
                3,
                8,
            ),
            "per_device_train_batch_size": trial.suggest_categorical(
                "per_device_train_batch_size",
                [8, 16],
            ),
            "weight_decay": trial.suggest_float(
                "weight_decay",
                0.0,
                0.1,
            ),
            "warmup_ratio": trial.suggest_float(
                "warmup_ratio",
                0.05,
                0.2,
            ),
        }

    def compute_objective(metrics):
        """
        Metric dùng để chọn best trial.

        Trainer sẽ thêm prefix 'eval_' vào metric từ compute_metrics.
        Nếu compute_metrics trả về 'f1_macro',
        ở đây sẽ nhận được 'eval_f1_macro'.
        """

        print("Metric keys from trial:", list(metrics.keys()))

        if "eval_f1_macro" in metrics:
            return metrics["eval_f1_macro"]

        if "eval_f1" in metrics:
            print("WARNING: Không thấy eval_f1_macro, fallback sang eval_f1.")
            return metrics["eval_f1"]

        if "eval_loss" in metrics:
            print("WARNING: Không thấy F1 metric, fallback sang -eval_loss.")
            return -metrics["eval_loss"]

        raise KeyError(
            "Không tìm thấy eval_f1_macro / eval_f1 / eval_loss trong metrics. "
            f"Metrics hiện có: {metrics}"
        )

    best_run = trainer_hp.hyperparameter_search(
        direction="maximize",
        backend="optuna",
        n_trials=HP_N_TRIALS,
        hp_space=hp_space,
        compute_objective=compute_objective,

        # Giảm rủi ro hết RAM GPU sau mỗi trial.
        gc_after_trial=True,
    )

    print("\n========== BEST RUN ==========")
    print(best_run)

    print("\n========== BEST HYPERPARAMETERS ==========")
    print(json.dumps(best_run.hyperparameters, indent=2))

    BEST_HP_PATH.parent.mkdir(parents=True, exist_ok=True)

    with open(BEST_HP_PATH, "w", encoding="utf-8") as f:
        json.dump(
            best_run.hyperparameters,
            f,
            ensure_ascii=False,
            indent=2,
        )

    print("\n✅ Saved best hyperparameters to:", BEST_HP_PATH)

In [ ]:
# ============================================================
# CELL C — Load best hyperparameters
# ============================================================

from pathlib import Path
import json

BEST_HP_PATH = Path(CFG["hp_dir"]) / "best_hyperparameters.json"

DEFAULT_HPS = {
    "learning_rate": 2e-5,
    "num_train_epochs": 3,
    "per_device_train_batch_size": 8,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
}

if BEST_HP_PATH.exists():
    with open(BEST_HP_PATH, "r", encoding="utf-8") as f:
        BEST_HPS = json.load(f)

    print("Loaded best HPs from Optuna:", BEST_HP_PATH)
else:
    BEST_HPS = DEFAULT_HPS
    print("Không thấy best_hyperparameters.json, dùng DEFAULT_HPS.")

# Chuẩn hóa type
BEST_HPS["learning_rate"] = float(BEST_HPS["learning_rate"])
BEST_HPS["num_train_epochs"] = int(BEST_HPS["num_train_epochs"])
BEST_HPS["per_device_train_batch_size"] = int(BEST_HPS["per_device_train_batch_size"])
BEST_HPS["weight_decay"] = float(BEST_HPS["weight_decay"])
BEST_HPS["warmup_ratio"] = float(BEST_HPS["warmup_ratio"])

print("\nBEST_HPS dùng cho final training:")
print(json.dumps(BEST_HPS, indent=2))

## Cell 21 — Chọn hyperparameters cho final training

Chức năng:
- Nếu đã chạy HP search, load best hyperparameters.
- Nếu chưa chạy, dùng baseline hyperparameters an toàn.

In [ ]:
# ============================================================
# CELL — Chọn bộ hyperparameters theo tên
# ============================================================
#
# Cách dùng:
# - Đổi SELECTED_HP_NAME thành tên bộ muốn chạy.
# - Nếu SELECTED_HP_NAME = "best_from_optuna" thì sẽ ưu tiên đọc BEST_HP_PATH.
# - Nếu chưa có BEST_HP_PATH thì fallback về baseline_current.

import json

MANUAL_HP_CONFIGS = {
    "baseline_current": {
        "learning_rate": 2e-5,
        "num_train_epochs": 3,
        "per_device_train_batch_size": 8,
        "weight_decay": 0.01,
        "warmup_ratio": 0.1,
    },

    "conservative_lr_low_epoch_5": {
        "learning_rate": 1e-5,
        "num_train_epochs": 5,
        "per_device_train_batch_size": 8,
        "weight_decay": 0.01,
        "warmup_ratio": 0.1,
    },

    "balanced_lr_3e5_decay_003": {
        "learning_rate": 3e-5,
        "num_train_epochs": 4,
        "per_device_train_batch_size": 8,
        "weight_decay": 0.03,
        "warmup_ratio": 0.1,
    },

    "aggressive_lr_5e5_batch_16": {
        "learning_rate": 5e-5,
        "num_train_epochs": 3,
        "per_device_train_batch_size": 16,
        "weight_decay": 0.05,
        "warmup_ratio": 0.15,
    },
}

# ============================================================
# ĐỔI TÊN BỘ Ở ĐÂY
# ============================================================

SELECTED_HP_NAME = "best_from_optuna"

# Các lựa chọn hợp lệ:
print("Available HP configs:")
for name in MANUAL_HP_CONFIGS.keys():
    print(" -", name)

print(" - best_from_optuna")

# ============================================================
# Load hyperparameters
# ============================================================

if SELECTED_HP_NAME == "best_from_optuna":
    if BEST_HP_PATH.exists():
        with open(BEST_HP_PATH, "r", encoding="utf-8") as f:
            BEST_HPS = json.load(f)

        print("\nLoaded best HPs from Optuna:", BEST_HP_PATH)
    else:
        print("\nBEST_HP_PATH chưa tồn tại, fallback về baseline_current.")
        BEST_HPS = MANUAL_HP_CONFIGS["baseline_current"]

else:
    if SELECTED_HP_NAME not in MANUAL_HP_CONFIGS:
        raise ValueError(
            f"SELECTED_HP_NAME không hợp lệ: {SELECTED_HP_NAME}\n"
            f"Hãy chọn một trong: {list(MANUAL_HP_CONFIGS.keys()) + ['best_from_optuna']}"
        )

    BEST_HPS = MANUAL_HP_CONFIGS[SELECTED_HP_NAME]
    print(f"\nUsing manual HP config: {SELECTED_HP_NAME}")

# ============================================================
# Chuẩn hóa type
# ============================================================

BEST_HPS = dict(BEST_HPS)

BEST_HPS["learning_rate"] = float(BEST_HPS["learning_rate"])
BEST_HPS["num_train_epochs"] = int(BEST_HPS["num_train_epochs"])
BEST_HPS["per_device_train_batch_size"] = int(BEST_HPS["per_device_train_batch_size"])
BEST_HPS["weight_decay"] = float(BEST_HPS["weight_decay"])
BEST_HPS["warmup_ratio"] = float(BEST_HPS["warmup_ratio"])

print("\nSelected BEST_HPS:")
print(json.dumps(BEST_HPS, indent=2))

## Cell 22 — Final training

Chức năng:
- Train full model với hyperparameters đã chọn.
- Chọn best checkpoint theo `f1_macro`.
- Bật early stopping để tránh overfit.

In [ ]:
final_args = TrainingArguments(
    output_dir=CFG["output_dir"],
    learning_rate=float(BEST_HPS["learning_rate"]),
    per_device_train_batch_size=int(BEST_HPS["per_device_train_batch_size"]),
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=1,
    num_train_epochs=int(BEST_HPS["num_train_epochs"]),
    weight_decay=float(BEST_HPS["weight_decay"]),
    warmup_ratio=float(BEST_HPS["warmup_ratio"]),

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    logging_steps=50,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED,
)

trainer = Trainer(
    model_init=model_init,
    args=final_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

val_metrics = trainer.evaluate(tokenized_ds["validation"])
print("Validation metrics:")
print(json.dumps(val_metrics, indent=2))

## Cell 23 — Đánh giá test set một lần cuối

Chức năng:
- Evaluate trên test set sau khi đã chọn xong model/hyperparameters.
- Không nên dùng test set để tune tiếp.

In [ ]:
if RUN_FINAL_TEST:
    test_metrics = trainer.evaluate(tokenized_ds["test"])
    print("Test metrics:")
    print(json.dumps(test_metrics, indent=2))

    Path(CFG["output_dir"]).mkdir(parents=True, exist_ok=True)
    with open(Path(CFG["output_dir"]) / "test_metrics.json", "w", encoding="utf-8") as f:
        json.dump(test_metrics, f, ensure_ascii=False, indent=2)
else:
    print("RUN_FINAL_TEST=False, bỏ qua test evaluation.")

## Cell 24 — Xem bảng metric theo entity type

Chức năng:
- Biến metric dict thành bảng dễ đọc.
- Tập trung vào `PER`, `ADDR`, `NOTE`.

In [ ]:
metrics_source = test_metrics if RUN_FINAL_TEST else val_metrics

rows = []
for ent_type in ENTITY_TYPES:
    rows.append({
        "entity": ent_type,
        "precision": metrics_source.get(f"eval_precision_{ent_type}", None),
        "recall": metrics_source.get(f"eval_recall_{ent_type}", None),
        "f1": metrics_source.get(f"eval_f1_{ent_type}", None),
        "support": metrics_source.get(f"eval_support_{ent_type}", None),
    })

entity_metric_df = pd.DataFrame(rows)
display(entity_metric_df)

print("Micro F1:", metrics_source.get("eval_f1"))
print("Macro F1:", metrics_source.get("eval_f1_macro"))

## Cell 25 — Hàm encode inference thống nhất

Chức năng:
- Chuẩn bị input cho inference.
- Trả thêm `token_spans` để gom token prediction thành entity span trên raw text.
- Hỗ trợ cả XLM-R và PhoBERT.

In [ ]:
def encode_for_inference(text, return_tensors="pt"):
    """
    Trả về:
    - enc: input model
    - token_spans: list span raw text cho từng token; special/subword phụ có thể là None
    """
    if MODEL_FAMILY == "xlmr":
        enc = tokenizer(
            text,
            return_offsets_mapping=True,
            return_tensors=return_tensors,
            truncation=True,
            max_length=MAX_LENGTH,
        )

        offsets = enc.pop("offset_mapping")[0].tolist()
        token_spans = [
            (int(s), int(e)) if int(e) > int(s) else None
            for s, e in offsets
        ]
        return enc, token_spans

    if MODEL_FAMILY == "phobert":
        words, word_spans = segment_with_spans(text)
        max_content_len = MAX_LENGTH - tokenizer.num_special_tokens_to_add(pair=False)

        flat_input_ids = []
        flat_token_spans = []

        for word, span in zip(words, word_spans):
            sub_ids = tokenizer.encode(word, add_special_tokens=False)
            if not sub_ids:
                continue

            if len(flat_input_ids) + len(sub_ids) > max_content_len:
                break

            flat_input_ids.extend(sub_ids)
            # Chỉ first subword đại diện cho word span.
            flat_token_spans.append(span)
            flat_token_spans.extend([None] * (len(sub_ids) - 1))

        input_ids = tokenizer.build_inputs_with_special_tokens(flat_input_ids)
        special_mask = tokenizer.get_special_tokens_mask(
            flat_input_ids,
            already_has_special_tokens=False,
        )

        token_spans = []
        j = 0
        for is_special in special_mask:
            if is_special:
                token_spans.append(None)
            else:
                token_spans.append(flat_token_spans[j])
                j += 1

        if return_tensors == "pt":
            enc = {
                "input_ids": torch.tensor([input_ids], dtype=torch.long),
                "attention_mask": torch.tensor([[1] * len(input_ids)], dtype=torch.long),
            }
        else:
            enc = {
                "input_ids": [input_ids],
                "attention_mask": [[1] * len(input_ids)],
            }

        return enc, token_spans

    raise ValueError(f"Unknown MODEL_FAMILY: {MODEL_FAMILY}")

## Cell 26 — Gom BIO token labels thành entity spans

Chức năng:
- Convert output token-level thành entity-level.
- Sửa nhẹ BIO lỗi ở inference bằng `ensure_iob2`.
- Trả về text span gốc, type, start, end.

In [ ]:
def bio_to_entities(text, labels, token_spans):
    """Gom chuỗi BIO labels thành entity spans trên raw text."""
    labels = ensure_iob2(labels)

    entities = []
    current = None

    def close_current():
        nonlocal current
        if current is not None:
            current["text"] = text[current["start"]:current["end"]]
            entities.append(current)
            current = None

    for lab, span in zip(labels, token_spans):
        if span is None:
            continue

        if lab == "O":
            close_current()
            continue

        prefix, ent_type = lab.split("-", 1)
        s, e = span

        if prefix == "B":
            close_current()
            current = {"label": ent_type, "start": s, "end": e}
        elif prefix == "I":
            if current is None or current["label"] != ent_type:
                close_current()
                current = {"label": ent_type, "start": s, "end": e}
            else:
                current["end"] = max(current["end"], e)

    close_current()
    return entities

def predict_entities(text, model=None):
    """Inference PyTorch model và trả về entities."""
    model = model or trainer.model
    model.eval()

    enc, token_spans = encode_for_inference(text, return_tensors="pt")
    device = next(model.parameters()).device
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        logits = model(**enc).logits[0].detach().cpu().numpy()

    pred_ids = logits.argmax(axis=-1).tolist()
    pred_labels = [id2label[int(i)] for i in pred_ids]

    return bio_to_entities(text, pred_labels, token_spans)

## Cell 27 — Inference thử

Chức năng:
- Test nhanh model bằng vài câu thực tế.
- Kiểm tra output entity-level thay vì chỉ token-level.

In [ ]:
examples = [
    "Nguyễn Văn A, 45 Lê Lợi, Quận 1, TP. Hồ Chí Minh, giao giờ hành chính",
    "Chị Mai ở KĐT Vinhomes Smart City, phường Tây Mỗ, Nam Từ Liêm, Hà Nội",
    "Anh Bình nhận hàng tại 12 Nguyễn Trãi, Thanh Xuân, ghi chú gọi trước 30 phút",
]

for text in examples:
    print("=" * 100)
    print(text)
    print(predict_entities(text))

## Cell 28 — Lưu final model

Chức năng:
- Lưu best model từ Trainer.
- Lưu tokenizer.
- Lưu label map và config.

In [ ]:
FINAL_DIR = Path(CFG["output_dir"])
FINAL_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(FINAL_DIR))
tokenizer.save_pretrained(str(FINAL_DIR))

with open(FINAL_DIR / "label2id.json", "w", encoding="utf-8") as f:
    json.dump(label2id, f, ensure_ascii=False, indent=2)

with open(FINAL_DIR / "id2label.json", "w", encoding="utf-8") as f:
    json.dump({str(k): v for k, v in id2label.items()}, f, ensure_ascii=False, indent=2)

with open(FINAL_DIR / "training_config.json", "w", encoding="utf-8") as f:
    json.dump({
        "model_family": MODEL_FAMILY,
        "model_name": MODEL_NAME,
        "max_length": MAX_LENGTH,
        "entity_types": ENTITY_TYPES,
        "best_hyperparameters": BEST_HPS,
    }, f, ensure_ascii=False, indent=2)

print("Saved final model to:", FINAL_DIR)

## Cell 29 — Export ONNX FP32

Chức năng:
- Convert PyTorch checkpoint sang ONNX.
- Lưu tokenizer cùng thư mục ONNX.

In [ ]:
# ============================================================
# CELL 29 — Export best/final model sang ONNX FP32
# ============================================================

from pathlib import Path
import shutil
from optimum.onnxruntime import ORTModelForTokenClassification

# ------------------------------------------------------------
# 1. Debug nhanh CFG
# ------------------------------------------------------------

print("CFG keys hiện có:")
for k in CFG.keys():
    print(" -", k)

# ------------------------------------------------------------
# 2. Lấy FINAL_DIR và ONNX_FP32_DIR an toàn hơn
# ------------------------------------------------------------

FINAL_DIR = Path(
    CFG.get(
        "final_model_dir",
        CFG.get(
            "final_dir",
            CFG["output_dir"]
        )
    )
)

ONNX_FP32_DIR = Path(
    CFG.get(
        "onnx_fp32_dir",
        str(WORKING_ROOT / "onnx_fp32")
    )
)

print("\nFINAL_DIR    :", FINAL_DIR)
print("ONNX_FP32_DIR:", ONNX_FP32_DIR)

# ------------------------------------------------------------
# 3. Nếu FINAL_DIR chưa tồn tại, thử lưu model hiện tại
# ------------------------------------------------------------
#
# Trường hợp bạn vừa train xong nhưng chưa save_pretrained,
# ta sẽ lưu trainer.model vào FINAL_DIR.

if not FINAL_DIR.exists():
    print("\nFINAL_DIR chưa tồn tại. Đang thử lưu model hiện tại...")

    FINAL_DIR.mkdir(parents=True, exist_ok=True)

    if "trainer" in globals():
        trainer.save_model(str(FINAL_DIR))
        tokenizer.save_pretrained(str(FINAL_DIR))
        print("Đã save trainer.model vào:", FINAL_DIR)

    elif "final_trainer" in globals():
        final_trainer.save_model(str(FINAL_DIR))
        tokenizer.save_pretrained(str(FINAL_DIR))
        print("Đã save final_trainer.model vào:", FINAL_DIR)

    elif "model" in globals():
        model.save_pretrained(str(FINAL_DIR))
        tokenizer.save_pretrained(str(FINAL_DIR))
        print("Đã save model vào:", FINAL_DIR)

    else:
        raise FileNotFoundError(
            "Không tìm thấy FINAL_DIR và cũng không thấy biến trainer/final_trainer/model để save.\n"
            "Bạn cần chạy cell final training trước."
        )

# ------------------------------------------------------------
# 4. Kiểm tra file bắt buộc
# ------------------------------------------------------------

if not (FINAL_DIR / "config.json").exists():
    raise FileNotFoundError(f"Thiếu config.json trong FINAL_DIR: {FINAL_DIR}")

has_weights = (
    (FINAL_DIR / "pytorch_model.bin").exists()
    or (FINAL_DIR / "model.safetensors").exists()
)

if not has_weights:
    raise FileNotFoundError(
        f"Không tìm thấy model weights trong {FINAL_DIR}.\n"
        "Cần có pytorch_model.bin hoặc model.safetensors."
    )

# ------------------------------------------------------------
# 5. Tạo lại thư mục ONNX output
# ------------------------------------------------------------

if ONNX_FP32_DIR.exists():
    shutil.rmtree(ONNX_FP32_DIR)

ONNX_FP32_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 6. Export PyTorch model -> ONNX FP32
# ------------------------------------------------------------

ort_model = ORTModelForTokenClassification.from_pretrained(
    str(FINAL_DIR),
    export=True,
)

ort_model.save_pretrained(str(ONNX_FP32_DIR))
tokenizer.save_pretrained(str(ONNX_FP32_DIR))

# ------------------------------------------------------------
# 7. Copy thêm metadata nếu có
# ------------------------------------------------------------

for name in [
    "label2id.json",
    "id2label.json",
    "training_config.json",
    "dataset_stats.json",
]:
    src = FINAL_DIR / name
    if src.exists():
        dst = ONNX_FP32_DIR / name
        dst.write_text(src.read_text(encoding="utf-8"), encoding="utf-8")
        print("Copied:", name)

# ------------------------------------------------------------
# 8. Kiểm tra file ONNX
# ------------------------------------------------------------

onnx_files = sorted(ONNX_FP32_DIR.glob("*.onnx"))

if not onnx_files:
    raise RuntimeError(f"Không tìm thấy file .onnx trong {ONNX_FP32_DIR}")

print("\n✅ Export ONNX FP32 thành công.")
print("Saved ONNX FP32 to:", ONNX_FP32_DIR)

print("\nONNX files:")
for p in onnx_files:
    size_mb = p.stat().st_size / (1024 * 1024)
    print(f" - {p.name}: {size_mb:.2f} MB")

print("\nCác file trong ONNX_FP32_DIR:")
for p in sorted(ONNX_FP32_DIR.iterdir()):
    print(" -", p.name)


## Cell 30 — Dynamic INT8 quantization cho CPU

Chức năng:
- Quantize ONNX FP32 sang INT8 động.
- Không cần calibration dataset.
- Phù hợp bước đầu để deploy CPU.

Gợi ý:
- `avx2` thường an toàn cho CPU phổ biến.
- Nếu server hỗ trợ AVX512/VNNI thì có thể đổi config để tối ưu hơn.

In [ ]:
from optimum.onnxruntime import ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig

ONNX_INT8_DIR = Path(CFG["onnx_int8_dir"])
ONNX_INT8_DIR.mkdir(parents=True, exist_ok=True)

quantizer = ORTQuantizer.from_pretrained(str(ONNX_FP32_DIR))

qconfig = AutoQuantizationConfig.avx2(
    is_static=False,
    per_channel=False,
)

quantizer.quantize(
    save_dir=str(ONNX_INT8_DIR),
    quantization_config=qconfig,
)

# Lưu tokenizer/metadata vào thư mục INT8.
tokenizer.save_pretrained(str(ONNX_INT8_DIR))
for name in ["label2id.json", "id2label.json", "training_config.json"]:
    src = FINAL_DIR / name
    if src.exists():
        dst = ONNX_INT8_DIR / name
        dst.write_text(src.read_text(encoding="utf-8"), encoding="utf-8")

print("Saved ONNX INT8 to:", ONNX_INT8_DIR)

## Cell 31 — Benchmark CPU: PyTorch vs ONNX INT8

Chức năng:
- Đo latency trung bình trên CPU.
- Benchmark end-to-end model forward, không tính download/loading model.
- Dùng batch size 1, phù hợp clipboard/API realtime.

In [ ]:
from optimum.onnxruntime import ORTModelForTokenClassification

def benchmark_forward(fn, n_warmup=10, n_runs=100):
    """Đo latency trung bình milliseconds/call."""
    for _ in range(n_warmup):
        fn()

    start = time.perf_counter()
    for _ in range(n_runs):
        fn()
    end = time.perf_counter()

    return (end - start) * 1000 / n_runs

bench_text = "Nguyễn Văn A, 45 Lê Lợi, Quận 1, TP. Hồ Chí Minh, giao giờ hành chính"

# Chuẩn bị input CPU.
enc_cpu, _ = encode_for_inference(bench_text, return_tensors="pt")
enc_cpu = {k: v.cpu() for k, v in enc_cpu.items()}

# PyTorch CPU.
pt_cpu_model = AutoModelForTokenClassification.from_pretrained(str(FINAL_DIR)).to("cpu").eval()

def run_pytorch_cpu():
    with torch.no_grad():
        _ = pt_cpu_model(**enc_cpu).logits

# ONNX INT8 CPU.
ort_int8_model = ORTModelForTokenClassification.from_pretrained(
    str(ONNX_INT8_DIR),
    provider="CPUExecutionProvider",
)

def run_onnx_int8_cpu():
    with torch.no_grad():
        _ = ort_int8_model(**enc_cpu).logits

pt_ms = benchmark_forward(run_pytorch_cpu)
onnx_ms = benchmark_forward(run_onnx_int8_cpu)

bench_df = pd.DataFrame([
    {"runtime": "PyTorch CPU FP32", "ms_per_call": pt_ms},
    {"runtime": "ONNX Runtime CPU INT8", "ms_per_call": onnx_ms},
])

display(bench_df)
print("Speedup:", pt_ms / onnx_ms if onnx_ms > 0 else None)

## Cell 32 — Inference bằng ONNX INT8

Chức năng:
- Load model ONNX INT8.
- Chạy inference entity-level tương tự PyTorch.

In [ ]:
def predict_entities_onnx_int8(text):
    """Inference bằng ONNX INT8 và gom entity spans."""
    enc, token_spans = encode_for_inference(text, return_tensors="pt")
    enc = {k: v.cpu() for k, v in enc.items()}

    outputs = ort_int8_model(**enc)
    logits = outputs.logits[0].detach().cpu().numpy()

    pred_ids = logits.argmax(axis=-1).tolist()
    pred_labels = [id2label[int(i)] for i in pred_ids]

    return bio_to_entities(text, pred_labels, token_spans)

for text in examples:
    print("=" * 100)
    print(text)
    print(predict_entities_onnx_int8(text))

## Cell 33 — Zip artifacts để tải từ Kaggle

Chức năng:
- Nén final PyTorch model.
- Nén ONNX FP32.
- Nén ONNX INT8.
- Lưu toàn bộ file zip vào `/kaggle/working/artifacts` để tải ở panel **Output/Files** của Kaggle.


In [ ]:
import shutil
from pathlib import Path
from IPython.display import display, FileLink

ARTIFACT_ROOT = WORKING_ROOT / "artifacts"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

final_zip = shutil.make_archive(
    str(ARTIFACT_ROOT / f"{MODEL_FAMILY}_ner_final_pytorch"),
    "zip",
    root_dir=str(FINAL_DIR),
)

onnx_fp32_zip = shutil.make_archive(
    str(ARTIFACT_ROOT / f"{MODEL_FAMILY}_ner_onnx_fp32"),
    "zip",
    root_dir=str(ONNX_FP32_DIR),
)

onnx_int8_zip = shutil.make_archive(
    str(ARTIFACT_ROOT / f"{MODEL_FAMILY}_ner_onnx_int8"),
    "zip",
    root_dir=str(ONNX_INT8_DIR),
)

print("Created:")
print(final_zip)
print(onnx_fp32_zip)
print(onnx_int8_zip)

print("\nTrên Kaggle, có thể tải các file này trong panel Output/Files:")
print(ARTIFACT_ROOT)

# FileLink thường hiển thị link download trực tiếp ngay dưới output cell.
for zip_path in [final_zip, onnx_fp32_zip, onnx_int8_zip]:
    display(FileLink(zip_path))


## Cell 34 — Checklist sau khi chạy

Sau khi notebook chạy xong, hãy kiểm tra:

1. `eval_f1_macro` trên validation/test.
2. `f1_NOTE`, `recall_NOTE` có bị quá thấp không.
3. Inference samples có gom đúng entity-level không.
4. ONNX INT8 latency có nhanh hơn PyTorch CPU không.
5. Nếu XLM-R đã ổn, đổi `MODEL_FAMILY = "phobert"` và chạy lại để so sánh công bằng.